<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch12_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8 - Build a CNN to achieve highest possible accuracy on MNIST


In [1]:
from sklearn.datasets import fetch_openml
import numpy as np
mnist = fetch_openml('mnist_784', as_frame = False)

X,y = mnist.data, mnist.target


In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape, y_test.shape

((56000, 784), (56000,), (7000, 784), (7000,), (7000, 784), (7000,))

In [32]:
X_train[0].shape

(784,)

In [33]:
y_train[:10]

array(['5', '4', '8', '0', '2', '6', '5', '4', '8', '3'], dtype=object)

In [34]:
# reshape the images
X_train = X_train.reshape(56000, 28,28)
X_val = X_val.reshape(7000, 28,28)
X_test = X_test.reshape(7000, 28,28)
X_train.shape

(56000, 28, 28)

In [35]:
# transform to tensors
import torch
import numpy as np
X_train = torch.tensor(X_train, dtype=torch.float32)/255
X_val = torch.tensor(X_val, dtype=torch.float32)/255
X_test = torch.tensor(X_test, dtype=torch.float32)/255

y_train = torch.tensor(y_train.astype(int), dtype=torch.long)
y_val   = torch.tensor(y_val.astype(int), dtype=torch.long)
y_test  = torch.tensor(y_test.astype(int), dtype=torch.long)




In [36]:
# let's add the channel dimension before the spatial dimensions
X_train = X_train.reshape(56000, 1, 28, 28)
X_val = X_val.reshape(7000, 1, 28,28)
X_test = X_test.reshape(7000, 1, 28,28)
X_train.shape

torch.Size([56000, 1, 28, 28])

In [38]:
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

# to use the DataLoader we need to wrap training data (X,y) into a dataset

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

n_batch=32

train_loader = DataLoader(train_dataset, batch_size=n_batch, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=n_batch, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=n_batch, shuffle=False)

In [39]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### Let's build an inception module

In [40]:
import torch.nn as nn
from functools import partial
import torch.nn.functional as F

class InceptionModule(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()
    DeafultConv2d = partial(nn.Conv2d,
                            stride=1, padding="same", bias=False
                            )
    DefaultMaxpool = partial(nn.MaxPool2d,
                             stride=1, padding=n7//2)
    self.column1 = DeafultConv2d(in_channels = in_channels,
                                 out_channels = in_channels//4,
                                 kernel_size=n1)
    self.column2 = nn.Sequential(
        DeafultConv2d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n5),
        nn.ReLU(),
        DeafultConv2d(in_channels = in_channels//4,
                      out_channels = in_channels//4,
                      kernel_size=n2)
    )
    self.column3 = nn.Sequential(
        DeafultConv2d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n6),
        nn.ReLU(),
        DeafultConv2d(in_channels = in_channels//4,
                      out_channels = in_channels//4,
                      kernel_size=n3)
    )
    self.column4 = nn.Sequential(
        DefaultMaxpool(kernel_size=n7),
        DeafultConv2d(in_channels = in_channels,
                      out_channels = in_channels//4,
                      kernel_size=n4)
    )

  def forward(self, inputs):
    out1 = self.column1(inputs)
    out2 = self.column2(inputs)
    out3 = self.column3(inputs)
    out4 = self.column4(inputs)

    combined = torch.cat((out1, out2, out3, out4), dim=1)
    return F.relu(combined)

## Let's build a SE-Inception module

In [41]:
# We build a SE block attached to an inception module

class SEInceptionModule(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7, bottleneck):
    super().__init__()
    # our inception module outpust in_channel many feature maps
    self.inception = InceptionModule(in_channels, n1, n2, n3, n4, n5, n6, n7)
    self.global_pool = nn.AdaptiveAvgPool2d(1)
    self.main = nn.Sequential(
        self.global_pool,
        nn.Flatten(),
        nn.Linear(in_channels, bottleneck),
        nn.ReLU(),
        nn.Linear(bottleneck, in_channels),
        nn.Sigmoid()
    )

  def forward(self, inputs):
    inception_output = self.inception(inputs)
    multipliers = self.main(inception_output) #tensor of the form [batch_size, in_channels]
    multipliers = multipliers.reshape(multipliers.shape[0], multipliers.shape[1], 1, 1)
    return inception_output * multipliers


## New CNN from scratch

In [42]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    DeafultConv2d = partial(nn.Conv2d,
                            padding="same", bias=False
                            )
    self.conv1 = nn.Sequential(
        DeafultConv2d(in_channels = 1,
                      out_channels = 16,
                      kernel_size=3),
        nn.BatchNorm2d(16),
        nn.ReLU()
        ) # image is now (16, 28, 28)
    self.se1 = SEInceptionModule(16,
                                 1, 3, 5, 1,
                                    1, 1, 3,
                                 bottleneck=4
                                  ) #image is now (16, 28, 28)
    self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2) #image is now (16, 14, 14)
    self.conv2 = nn.Sequential(
        DeafultConv2d(in_channels = 16,
                      out_channels = 64,
                      kernel_size=3),
        nn.BatchNorm2d(64),
        nn.ReLU()
        ) # image is now (64, 14, 14)
    self.se2 = SEInceptionModule(64,
                                 1, 3, 5, 1,
                                    1, 1, 3,
                                 bottleneck=16
                                  ) # image is now (64, 14, 14)
    self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2) #image is now (64, 7, 7)
    self.global_pool = nn.AdaptiveAvgPool2d(1) # image is now (64, 1, 1)
    # here we need to flatten
    self.linear = nn.Linear(64, 10)

    self.cnn = nn.Sequential(
        self.conv1,
        self.se1,
        self.maxpool1,
        self.conv2,
        self.se2,
        self.maxpool2,
        self.global_pool,
        nn.Flatten(),
        self.linear
    )

  def forward(self, inputs):
    return self.cnn(inputs)


In [43]:
# now let's copy all the necessary machinery to train this model
%pip install torchmetrics

In [44]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch) #update at each iteration
  return metric.compute()

In [45]:
import time
def train_with_early_stopping(model, optimizer, criterion, metric,
          train_loader, valid_loader, n_epochs,
            patience=10, checkpoint_path=None, scheduler = None):
  checkpoint_path = checkpoint_path or "my_checkpoint.pt"

  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  best_metric = 0.0 #we save the current best validation metric
  patience_counter = 0

  for epoch in range(n_epochs):

    # We MUST force the model back into training mode at the start of every epoch.
    model.train()

    # We wipe the metric memory clean before the new epoch starts.
    metric.reset()

    total_loss = 0
    t0=time.time()

    for X_batch, y_batch in train_loader:

      # Send data to GPU/CPU memory
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      # Forward pass: get model predictions
      y_pred = model(X_batch)

      # Calculate how wrong the model is (Loss value)
      loss = criterion(y_pred, y_batch)

      # We use '.item()' to extract the raw Python number from the loss tensor.
      total_loss += loss.item()

      # Backpropagation: Calculate gradients and update model parameters
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      # We feed the current batch's predictions and targets into the tracker.
      metric.update(y_pred, y_batch)

    # Calculate average loss for this training epoch
    mean_loss = total_loss / len(train_loader)
    history["train_losses"].append(mean_loss)
    train_metric = metric.compute().item()
    valid_metric = evaluate_tm(model, valid_loader, metric).item()
    if valid_metric > best_metric:
          torch.save(model.state_dict(), checkpoint_path)
          best_metric = valid_metric
          best = " (best)" #print (best) next to the validation metric if better
          patience_counter = 0
    else:
          patience_counter += 1
          best = ""

    t1 = time.time()
    history["train_metrics"].append(train_metric)
    history["valid_metrics"].append(valid_metric)
    print(f"Epoch {epoch + 1}/{n_epochs}, "
          f"train loss: {history['train_losses'][-1]:.4f}, "
          f"train metric: {history['train_metrics'][-1]:.4f}, "
          f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
          f" in {t1 - t0:.1f}s"
        )
    if scheduler is not None:
      # change the learning rate according to the scheduler's rule
          scheduler.step()
    if patience_counter >= patience:
            print("Early stopping!")
            break
  # Reload the highest-accuracy weights so the model doesn't
  # stick with the worse, overfitted weights from the final epoch.
  # Because we are 10 epochs past the best model

  model.load_state_dict(torch.load(checkpoint_path))
  return history

In [48]:
torch.manual_seed(42)
model = CNN().to(device)
optimizer = torch.optim.NAdam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device) #n_classes = 10
n_epochs = 20
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, val_loader, n_epochs, patience =5)

Epoch 1/20, train loss: 0.3096, train metric: 0.9003, valid metric: 0.9634 (best) in 12.8s
Epoch 2/20, train loss: 0.0819, train metric: 0.9749, valid metric: 0.9766 (best) in 12.4s
Epoch 3/20, train loss: 0.0637, train metric: 0.9803, valid metric: 0.9786 (best) in 13.2s
Epoch 4/20, train loss: 0.0526, train metric: 0.9835, valid metric: 0.9847 (best) in 13.3s
Epoch 5/20, train loss: 0.0458, train metric: 0.9857, valid metric: 0.9814 in 12.5s
Epoch 6/20, train loss: 0.0386, train metric: 0.9876, valid metric: 0.9877 (best) in 12.7s
Epoch 7/20, train loss: 0.0362, train metric: 0.9882, valid metric: 0.9850 in 12.5s
Epoch 8/20, train loss: 0.0321, train metric: 0.9893, valid metric: 0.9901 (best) in 13.5s
Epoch 9/20, train loss: 0.0276, train metric: 0.9917, valid metric: 0.9867 in 12.4s
Epoch 10/20, train loss: 0.0261, train metric: 0.9913, valid metric: 0.9870 in 12.8s
Epoch 11/20, train loss: 0.0233, train metric: 0.9927, valid metric: 0.9859 in 12.7s
Epoch 12/20, train loss: 0.0222,

In [49]:
#let's evaluate on the test set!
print(f"Test accuracy of our model: {evaluate_tm(model, test_loader, accuracy)}")
# Test accuracy of our model: 0.9915714263916016

Test accuracy of our model: 0.9915714263916016


# 9 Transfer Learning for Image classification

In [50]:
from pathlib import Path
import urllib.request
import zipfile

root = "/content/drive/MyDrive/Colab Notebooks"

def download_hymenoptera_dataset():
    data_dir = Path(root)
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    zip_path = data_dir / "hymenoptera_data.zip"
    data_dir.mkdir(parents=True, exist_ok=True)
    if not zip_path.exists():
        print("Downloading...",  end="")
        urllib.request.urlretrieve(url, zip_path)
        print(" Done.")
    unzipped_dir = data_dir / "hymenoptera_data"
    if not unzipped_dir.exists():
        print("Extracting...", end="")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(data_dir)
        print(" Done.")
    return unzipped_dir

hymenoptera_dir = download_hymenoptera_dataset()

Downloading... Done.
Extracting... Done.


The problem we’re going to solve today is to train a model to classify ants and bees. We have about 120 training images each for ants and bees. There are 75 validation images for each class. Usually, this is a very small dataset to generalize upon, if trained from scratch. Since we are using transfer learning, we should be able to generalize reasonably well.

In [71]:
from torchvision.datasets import ImageFolder
import torchvision.transforms.v2 as T

# data augmentation pipeline
transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# no augmentation here
valid_test_transforms = T.Compose([
    T.Resize(232),
    T.CenterCrop(224),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_set9 = ImageFolder(hymenoptera_dir /  "train", transforms)
valid_test_set9 = ImageFolder(hymenoptera_dir / "val", valid_test_transforms)

torch.manual_seed(42)
valid_set9, test_set9 = torch.utils.data.random_split(valid_test_set9, [0.5, 0.5])


In [72]:
sample_image, sample_label = train_set9[0]
sample_image.shape, sample_label

(torch.Size([3, 224, 224]), 0)

In [73]:
train_set9.classes

['ants', 'bees']

In [74]:
classes=train_set9.classes

In [75]:
# let's create the dataloaders
train_loader9 = DataLoader(train_set9, batch_size=32, shuffle=True)
valid_loader9 = DataLoader(valid_set9, batch_size=32, shuffle=False)
test_loader9 = DataLoader(test_set9, batch_size=32, shuffle=False)

In [83]:
# let's import the model
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model_ex9 = torchvision.models.convnext_base(weights=weights).to(device)

In [84]:
model_ex9.classifier

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)

In [85]:
n_classes9=2

#we change the last layer of the model's classifier
model_ex9.classifier[2] = nn.Linear(1024, n_classes9).to(device)

In [92]:
## we freeze all the parameters except the one of the last layer
for param in model_ex9.parameters():
  param.requires_grad = False

for param in model_ex9.classifier.parameters():
  param.requires_grad = True


In [94]:
torch.manual_seed(42)
model_ex9 = model_ex9.to(device)
optimizer = torch.optim.NAdam(model_ex9.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(device)
n_epochs = 5
history = train_with_early_stopping(model_ex9, optimizer, criterion, accuracy,
                                    train_loader9, valid_loader9, n_epochs, patience =5)

Epoch 1/5, train loss: 0.0768, train metric: 0.9877, valid metric: 1.0000 (best) in 7.9s
Epoch 2/5, train loss: 0.0575, train metric: 0.9877, valid metric: 1.0000 in 7.8s
Epoch 3/5, train loss: 0.0380, train metric: 0.9918, valid metric: 1.0000 in 7.0s
Epoch 4/5, train loss: 0.0346, train metric: 0.9877, valid metric: 1.0000 in 7.6s
Epoch 5/5, train loss: 0.0306, train metric: 0.9918, valid metric: 1.0000 in 7.1s


In [95]:
print(f"Accuracy on the test set: {evaluate_tm(model_ex9, test_loader9, accuracy)}")

Accuracy on the test set: 0.9868420958518982
